# 🐍 Day 14 · Python Basics — map, filter, reduce & Parallel Work (Map-Reduce)

> **Goal of this notebook:** Map-reduce means *split the work, do each piece (map), then combine (reduce)*. Learn the Python tools and run pieces in parallel with threads.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. `map()` and `filter()`
2. `reduce()` and `operator.add`
3. `Annotated` reducers in LangGraph
4. Map-reduce over document pages
5. Running tasks in parallel with threads
6. Timing your code

## 1. `map()` and `filter()`
- `map(func, items)` → apply a function to every item.
- `filter(func, items)` → keep items where the function returns `True`.

In [ ]:
pages = ["Revenue grew 10%", "", "Net income was $50B", "   ", "Risks include inflation"]

lengths = list(map(len, pages))
non_empty = list(filter(lambda p: p.strip() != "", pages))

print(lengths)
print(non_empty)

# The same with list comprehensions (often easier to read):
print([len(p) for p in pages])
print([p for p in pages if p.strip()])

## 2. `reduce()` and `operator.add`
`reduce` combines a list into **one** value, two items at a time.

In [ ]:
from functools import reduce
import operator

numbers = [3, 5, 2, 10]
total = reduce(operator.add, numbers)       # ((3+5)+2)+10
print(total)

print(operator.add([1, 2], [3]))            # for lists, + joins them: [1, 2, 3]

## 3. `Annotated` reducers in LangGraph
When many parallel nodes write to the **same** key, LangGraph needs to know how to combine them:
```python
summaries: Annotated[list, operator.add]    # 'join the lists together'
```

In [ ]:
from typing import Annotated, TypedDict

class State(TypedDict):
    summaries: Annotated[list, operator.add]

# What LangGraph does when two nodes each return {"summaries": [...]}
current = {"summaries": ["summary of page 1"]}
update  = {"summaries": ["summary of page 2"]}
current["summaries"] = operator.add(current["summaries"], update["summaries"])
print(current)

## 4. 🔗 Map-reduce over pages

In [ ]:
def summarise(page):                    # MAP step (a real one calls an LLM)
    return page.split(".")[0][:40]

def combine(summaries):                  # REDUCE step
    return " | ".join(summaries)

doc_pages = [
    "JPMorgan revenue was $158B. It grew because of interest rates.",
    "Net income reached $49.6B. This was a record year.",
    "Main risks are credit losses. Regulation also matters.",
]

mapped = [summarise(p) for p in doc_pages]
print("MAP   :", mapped)
print("REDUCE:", combine(mapped))

## 5. Running tasks in parallel
LLM calls spend most of their time **waiting** for the internet. Threads let us wait for many at once.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def slow_summarise(page):
    time.sleep(1)                         # pretend the LLM takes 1 second
    return summarise(page)

start = time.time()
one_by_one = [slow_summarise(p) for p in doc_pages]
print(f"One by one: {time.time() - start:.1f} s")

start = time.time()
with ThreadPoolExecutor(max_workers=3) as pool:
    in_parallel = list(pool.map(slow_summarise, doc_pages))
print(f"In parallel: {time.time() - start:.1f} s")
print(in_parallel == one_by_one)          # same results, faster!

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Use `map` to make every word in `["rag", "llm"]` uppercase.
2. Use `filter` to keep only numbers bigger than 10 from `[5, 12, 8, 30]`.
3. Use `reduce` with `operator.mul` to multiply `[1, 2, 3, 4]`.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
print(list(map(str.upper, ["rag", "llm"])))
print(list(filter(lambda n: n > 10, [5, 12, 8, 30])))
print(reduce(operator.mul, [1, 2, 3, 4]))   # 24
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Map | `list(map(len, items))` |
| Filter | `list(filter(f, items))` |
| Reduce | `reduce(operator.add, nums)` |
| Reducer type | `Annotated[list, operator.add]` |
| Parallel | `ThreadPoolExecutor().map(f, items)` |
| Timing | `time.time()` |

## ➡️ What's next?
You now know the Python used in **Day_14_LangGraph_MapReduce_and_MultiAgent**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀